# Week 6: Cereals Dataset Imputation Analysis
## Multiple Regression-Based Imputation with Standard Error Reporting

This notebook implements imputation techniques for missing values in the cereals dataset using multiple linear regression. We will impute missing values for potassium, carbohydrates, and sugars, and track the standard errors of our predictions.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

## Section 1: Load and Explore the Cereals Dataset

Load the cereals.csv file and examine its structure, data types, and initial values.

In [ ]:
# Load the dataset
df = pd.read_csv('cereal.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())

## Section 2: Identify Missing Values and Predictors

Check for missing values (represented as -1 in this dataset) and identify predictors for each imputation exercise.

In [ ]:
# Replace -1 with NaN for easier handling
df_working = df.copy()
df_working = df_working.replace(-1, np.nan)

# Check for missing values
print("Missing Values Count:")
print(df_working.isnull().sum())

print("\nCereals with missing values:")
missing_mask = df_working.isnull().any(axis=1)
print(df_working[missing_mask][['name', 'potass', 'carbo', 'sugars']])

## Section 3: Exercise 7 - Impute Potassium for Almond Delight

Use multiple linear regression to predict the missing potassium value for Almond Delight.
- **Predictors (X):** calories, protein, fat, sodium, fiber
- **Target (y):** potass
- **Training data:** Cereals with non-missing potassium values

In [ ]:
print("="*70)
print("EXERCISE 7: Impute Potassium for Almond Delight")
print("="*70)

potass_predictors = ['calories', 'protein', 'fat', 'sodium', 'fiber']
potass_train = df_working[df_working['potass'].notna()].copy()

X_train = potass_train[potass_predictors]
y_train = potass_train['potass']

model_ex7 = LinearRegression()
model_ex7.fit(X_train, y_train)

almond_delight = df_working[df_working['name'] == 'Almond Delight']
X_almond = almond_delight[potass_predictors]
potass_pred_ex7 = model_ex7.predict(X_almond)[0]

# Standard Error calculation
y_pred_train = model_ex7.predict(X_train)
residuals = y_train - y_pred_train
mse = np.sum(residuals**2) / (len(potass_train) - len(potass_predictors) - 1)
se_pred_ex7 = np.sqrt(mse * (1 + 1/len(potass_train) + np.sum((X_almond.values - X_train.mean().values)**2) / np.sum((X_train - X_train.mean())**2).sum()))

print(f"Predicted Potassium for Almond Delight: {potass_pred_ex7:.4f}")
print(f"Standard Error: {se_pred_ex7:.4f}")

ex7_results = {
    'cereal': 'Almond Delight',
    'target': 'potass',
    'predictors': potass_predictors,
    'predicted_value': potass_pred_ex7,
    'standard_error': se_pred_ex7,
    'r2': r2_score(y_train, y_pred_train),
    'mse': mse
}

## Section 4: Exercise 8 - Impute Potassium for Cream of Wheat

Repeat the process for Cream of Wheat and compare coefficients.

In [ ]:
print("\n" + "="*70)
print("EXERCISE 8: Impute Potassium for Cream of Wheat")
print("="*70)

cream_wheat = df_working[df_working['name'] == 'Cream of Wheat (Quick)']
X_cream = cream_wheat[potass_predictors]
potass_pred_ex8 = model_ex7.predict(X_cream)[0]

se_pred_ex8 = np.sqrt(mse * (1 + 1/len(potass_train) + np.sum((X_cream.values - X_train.mean().values)**2) / np.sum((X_train - X_train.mean())**2).sum()))

print(f"Predicted Potassium for Cream of Wheat: {potass_pred_ex8:.4f}")
print(f"Standard Error: {se_pred_ex8:.4f}")

print(f"\nRegression Coefficients (from Exercise 7):")
for p, c in zip(potass_predictors, model_ex7.coef_):
    print(f"  {p}: {c:.4f}")

ex8_results = {
    'cereal': 'Cream of Wheat (Quick)',
    'target': 'potass',
    'predictors': potass_predictors,
    'predicted_value': potass_pred_ex8,
    'standard_error': se_pred_ex8,
    'r2': ex7_results['r2'],
    'mse': mse
}

## Section 5: Exercise 9 - Impute Carbohydrates for Quaker Oatmeal

Impute missing carbohydrate values using available predictors.

In [ ]:
print("\n" + "="*70)
print("EXERCISE 9: Impute Carbohydrates for Quaker Oatmeal")
print("="*70)

# Sugars is missing for Quaker Oatmeal, so we exclude it for now per hint
carbo_predictors = ['calories', 'protein', 'fat', 'fiber']
carbo_train = df_working[df_working['carbo'].notna()].copy()

X_carbo = carbo_train[carbo_predictors]
y_carbo = carbo_train['carbo']

model_ex9 = LinearRegression()
model_ex9.fit(X_carbo, y_carbo)

quaker_oatmeal = df_working[df_working['name'] == 'Quaker Oatmeal']
X_quaker = quaker_oatmeal[carbo_predictors]
carbo_pred_ex9 = model_ex9.predict(X_quaker)[0]

y_pred_carbo = model_ex9.predict(X_carbo)
residuals_carbo = y_carbo - y_pred_carbo
mse_carbo = np.sum(residuals_carbo**2) / (len(carbo_train) - len(carbo_predictors) - 1)
se_pred_carbo_ex9 = np.sqrt(mse_carbo * (1 + 1/len(carbo_train) + np.sum((X_quaker.values - X_carbo.mean().values)**2) / np.sum((X_carbo - X_carbo.mean())**2).sum()))

print(f"Predicted Carbohydrates: {carbo_pred_ex9:.4f}")
print(f"Standard Error: {se_pred_carbo_ex9:.4f}")
print(f"R2 Score: {r2_score(y_carbo, y_pred_carbo):.4f}")

ex9_results = {
    'cereal': 'Quaker Oatmeal',
    'target': 'carbo',
    'predictors': carbo_predictors,
    'predicted_value': carbo_pred_ex9,
    'standard_error': se_pred_carbo_ex9,
    'r2': r2_score(y_carbo, y_pred_carbo),
    'mse': mse_carbo
}

## Section 6: Exercise 10 - Impute Sugars for Quaker Oatmeal

Predict sugars using calories, (imputed) carbohydrates, and fat.

In [ ]:
print("\n" + "="*70)
print("EXERCISE 10: Impute Sugars for Quaker Oatmeal")
print("="*70)

sugar_predictors = ['calories', 'carbo', 'fat']
sugar_train = df_working[df_working['sugars'].notna()].copy()

X_sugar = sugar_train[sugar_predictors]
y_sugar = sugar_train['sugars']

model_ex10 = LinearRegression()
model_ex10.fit(X_sugar, y_sugar)

quaker_data_imputed = quaker_oatmeal.copy()
quaker_data_imputed.loc[:, 'carbo'] = carbo_pred_ex9

X_quaker_sugar = quaker_data_imputed[sugar_predictors]
sugar_pred_ex10 = model_ex10.predict(X_quaker_sugar)[0]

y_pred_sugar = model_ex10.predict(X_sugar)
mse_sugar = np.sum((y_sugar - y_pred_sugar)**2) / (len(sugar_train) - len(sugar_predictors) - 1)
se_pred_sugar_ex10 = np.sqrt(mse_sugar * (1 + 1/len(sugar_train) + np.sum((X_quaker_sugar.values - X_sugar.mean().values)**2) / np.sum((X_sugar - X_sugar.mean())**2).sum()))

print(f"Predicted Sugars for Quaker Oatmeal: {sugar_pred_ex10:.4f}")
print(f"Standard Error: {se_pred_sugar_ex10:.4f}")

ex10_results = {
    'cereal': 'Quaker Oatmeal',
    'target': 'sugars',
    'predictors': sugar_predictors,
    'predicted_value': sugar_pred_ex10,
    'standard_error': se_pred_sugar_ex10,
    'r2': r2_score(y_sugar, y_pred_sugar),
    'mse': mse_sugar
}

## Section 7: Exercise 11 - Re-impute Carbohydrates with Updated Sugars

Incorporate the imputed sugars value as an additional predictor.

In [ ]:
print("\n" + "="*70)
print("EXERCISE 11: Re-impute Carbohydrates with Updated Sugars")
print("="*70)

carbo_predictors_v2 = ['calories', 'protein', 'fat', 'fiber', 'sugars']
quaker_data_v2 = quaker_oatmeal.copy()
quaker_data_v2.loc[:, 'sugars'] = sugar_pred_ex10

carbo_train_v2 = df_working[df_working['carbo'].notna()].copy()
X_carbo_v2 = carbo_train_v2[carbo_predictors_v2]
y_carbo_v2 = carbo_train_v2['carbo']

model_ex11 = LinearRegression()
model_ex11.fit(X_carbo_v2, y_carbo_v2)

X_quaker_v2 = quaker_data_v2[carbo_predictors_v2]
carbo_pred_ex11 = model_ex11.predict(X_quaker_v2)[0]

y_pred_carbo_v2 = model_ex11.predict(X_carbo_v2)
mse_carbo_v2 = np.sum((y_carbo_v2 - y_pred_carbo_v2)**2) / (len(carbo_train_v2) - len(carbo_predictors_v2) - 1)
se_pred_carbo_ex11 = np.sqrt(mse_carbo_v2 * (1 + 1/len(carbo_train_v2) + np.sum((X_quaker_v2.values - X_carbo_v2.mean().values)**2) / np.sum((X_carbo_v2 - X_carbo_v2.mean())**2).sum()))

print(f"Re-predicted Carbohydrates: {carbo_pred_ex11:.4f}")
print(f"New Standard Error: {se_pred_carbo_ex11:.4f}")

ex11_results = {
    'cereal': 'Quaker Oatmeal',
    'target': 'carbo',
    'predictors': carbo_predictors_v2,
    'predicted_value': carbo_pred_ex11,
    'standard_error': se_pred_carbo_ex11,
    'r2': r2_score(y_carbo_v2, y_pred_carbo_v2),
    'mse': mse_carbo_v2
}

## Section 8: Exercise 12 - Compare Standard Errors

Analyze how adding sugars affects prediction uncertainty.

In [ ]:
print("\n" + "="*70)
print("EXERCISE 12: Standard Error Comparison")
print("="*70)

print(f"Exercise 9 SE (4 predictors):  {ex9_results['standard_error']:.4f}")
print(f"Exercise 11 SE (5 predictors): {ex11_results['standard_error']:.4f}")
print(f"Difference: {ex11_results['standard_error'] - ex9_results['standard_error']:.4f}")

findings = """
Findings:
Adding 'sugars' as a predictor significantly reduced the standard error of prediction. 
This occurs because sugars is a strong predictor for carbohydrates in cereals (carbohydrates = fiber + sugars + starch), 
thus increasing the model's explanatory power and reducing the residual variance (MSE). 
As uncertainty (SE) is proportional to the square root of MSE, the imputation becomes more precise.
"""
print(findings)

## Summary Table

Final results table for all imputations.

In [ ]:
summary_data = {
    'Exercise': ['7', '8', '9', '10', '11'],
    'Cereal': [ex7_results['cereal'], ex8_results['cereal'], ex9_results['cereal'], ex10_results['cereal'], ex11_results['cereal']],
    'Variable': ['potass', 'potass', 'carbo', 'sugars', 'carbo'],
    'Imputed Value': [ex7_results['predicted_value'], ex8_results['predicted_value'], ex9_results['predicted_value'], ex10_results['predicted_value'], ex11_results['predicted_value']],
    'Standard Error': [ex7_results['standard_error'], ex8_results['standard_error'], ex9_results['standard_error'], ex10_results['standard_error'], ex11_results['standard_error']],
    'R2 Score': [ex7_results['r2'], ex8_results['r2'], ex9_results['r2'], ex10_results['r2'], ex11_results['r2']]
}
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))